In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from utils import *
import prompt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import openai
import json
client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])
import matplotlib.pyplot as plt


# --- Load data ---

In [ ]:
config = load_config("configs/config_cluster_merfish.yaml")
config.data_name = "MERFISH_29"
config.refresh_paths()
name_truth = config.name_truth


In [ ]:
# --- Load data ---
data_path = str(dataset_file("merfish", config.data_name))
adata = sc.read_h5ad(data_path) 
# rename the column of cell_class to cell_type
adata.obs.rename(columns={'cell_class': 'cell_type'}, inplace=True)

# clean the cell ID to save token
adata.obs_names = list(range(len(adata)))
adata.obs_names = adata.obs_names.astype(str)

# Remove rows with NaN values in 'layer_guess'
adata = adata[~adata.obs[name_truth].isna()].copy()

# Verify that NaNs have been removed
remaining_nan_count = adata.obs[name_truth].isna().sum()
print(f"Remaining NaN values in '{name_truth}' after removal: {remaining_nan_count}")

pos_data = pd.DataFrame(adata.obsm['spatial'], columns=['x', 'y'], index=adata.obs_names)
adata.obs = adata.obs.join(pos_data)

# rename cell types
celltype_rename = {
    'Astrocyte' : 'Astrocyte',
 'Endothelial 1': 'Endothelial',
 'OD Mature 2': 'Mature oligodendrocytes',
 'Inhibitory': 'Inhibitory',
 'OD Immature 1': 'Immature oligodendrocytes',
 'Excitatory': 'Excitatory',
 'Endothelial 3': 'Endothelial',
 'Microglia': 'Microglia',
 'OD Mature 1': 'Mature oligodendrocytes',
 'Pericytes': 'Pericytes',
 'OD Mature 4': 'Mature oligodendrocytes',
 'Endothelial 2': 'Endothelial',
 'OD Mature 3': 'Mature oligodendrocytes',
 'OD Immature 2': 'Immature oligodendrocytes',
 'Ependymal': 'Ependymal'
}
adata.obs['cell_type'] = adata.obs['cell_type'].map(celltype_rename)
celltype_data = adata.obs[['cell_type']]

# normalize and scale the data
sc.pp.filter_genes(adata, min_cells=5)
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.scale(adata)


# --- Compute adjacency matrix ---
r = config.r
adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
# add diagonal to the adj_matrix
adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

# --- Generate one-hot encoded matrix ---
one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
one_hot_matrix = one_hot_df.values
one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

# --- Calculate neighbor counts ---
neighbor_count = adj_matrix.dot(one_hot_matrix)
n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
# Convert n_neighbors to a column vector for element-wise division
n_neighbors_col = n_neighbors.reshape(-1, 1)
# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized = neighbor_count / n_neighbors_col

neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                              index=celltype_data.index, 
                              columns=one_hot_df.columns.str.lstrip('_'))



In [ ]:
# if graph type is countplusgene
# Initialize a dictionary to store the top genes per cell type
top_genes_per_cell_type = {}

for cell_type in adata.obs['cell_type'].unique():
    # Subset the data for the current cell type
    adata_subset = adata[adata.obs['cell_type'] == cell_type].copy()
    if len(adata_subset) < 30:
        continue

    
    # Compute highly variable genes within the subset
    sc.pp.highly_variable_genes(
        adata_subset,
        n_top_genes=5,
        flavor='seurat',
        subset=False,
        layer=None,
        inplace=True
    )
    
    # Retrieve the top 5 highly variable genes
    top_genes = adata_subset.var.loc[adata_subset.var['highly_variable'], :].index.tolist()
    
    # Store the results in the dictionary
    top_genes_per_cell_type[cell_type] = top_genes

# Convert the dictionary to a DataFrame for better visualization
top_genes_df = pd.DataFrame.from_dict(top_genes_per_cell_type, orient='index').transpose()

# get all the top genes
top_genes = list(set(top_genes_df.values.flatten()))

# --- Calculate neighbor genes ---
neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

# Perform element-wise division between neighbor_count and n_neighbors_col
neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                              index=adata.obs_names, 
                              columns=top_genes)



# different r

In [ ]:
kmeansBoth_ari = []
kmeans_ari = []
kemans_genes_ari = []

for r in range(10, 1000, 10):
    adj_matrix, distances = sparse_adjacency(pos_data, threshold=r)
    # add diagonal to the adj_matrix
    adj_matrix = adj_matrix + scipy.sparse.diags(np.ones(adj_matrix.shape[0]))

    # --- Generate one-hot encoded matrix ---
    one_hot_df = pd.get_dummies(celltype_data['cell_type'], prefix='').astype(int)
    one_hot_matrix = one_hot_df.values
    one_hot_matrix = csr_matrix(one_hot_matrix)  # Convert to sparse matrix

    # --- Calculate neighbor counts ---
    neighbor_count = adj_matrix.dot(one_hot_matrix)
    n_neighbors = adj_matrix.sum(axis=1).A1  # .A1 converts to 1D numpy array
    # Convert n_neighbors to a column vector for element-wise division
    n_neighbors_col = n_neighbors.reshape(-1, 1)
    # Perform element-wise division between neighbor_count and n_neighbors_col
    neighbor_matrix_normalized = neighbor_count / n_neighbors_col

    neighbor_normalized_df = pd.DataFrame(neighbor_matrix_normalized.toarray(), 
                                index=celltype_data.index, 
                                columns=one_hot_df.columns.str.lstrip('_'))
    # --- Calculate neighbor genes ---
    neighbor_genes = adj_matrix.dot(adata[:,top_genes].X)

    # Perform element-wise division between neighbor_count and n_neighbors_col
    neighbor_matrix_normalized_genes = neighbor_genes / n_neighbors_col

    neighbor_normalized_df_genes = pd.DataFrame(neighbor_matrix_normalized_genes, 
                                index=adata.obs_names, 
                                columns=top_genes)
    
    neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()

    # kmeans with both neighbor count and neighbor genes
    km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
    clusters = km.fit_predict(neighbor_scaled_df)
    kmeansBoth_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))

    # kmeans with neighbor count
    km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
    clusters = km.fit_predict(neighbor_normalized_df)
    kmeans_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))

    # kmeans with neighbor genes
    km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
    clusters = km.fit_predict(neighbor_normalized_df_genes)
    kemans_genes_ari.append(adjusted_rand_score(adata.obs[config.name_truth], clusters))



In [ ]:
# plot r and ari of kmeans_ari, kemans_genes_ari and kmeansBoth_ari
plt.plot(range(10, 1000, 10), kmeansBoth_ari, label='kmeansBoth')
plt.plot(range(10, 1000, 10), kmeans_ari, label='kmeans')
plt.plot(range(10, 1000, 10), kemans_genes_ari, label='kmeans_genes')
plt.xlabel('r')
plt.ylabel('ARI')
plt.title(f'ARI of KMeans with different r for {config.data_name}')
plt.legend()
plt.show()





# cluster cells

In [ ]:
neighbor_scaled_df = neighbor_normalized_df.join(neighbor_normalized_df_genes).copy()



In [ ]:
neighbor_scaled_df

In [ ]:
# use KMeans instead of KModes

km = KMeans(n_clusters=len(adata.obs[config.name_truth].unique()), random_state=42)
clusters = km.fit_predict(neighbor_scaled_df)
cluster_centers = km.cluster_centers_
cluster_centers = pd.DataFrame(cluster_centers, columns=neighbor_scaled_df.columns)
adata.obs['kmeans'] = clusters.astype(str)

# plot initial clusters

In [ ]:
sc.pl.scatter(adata,x="x",y="y", color="kmeans", title =  f"{config.data_name} Kmeans cluster results")
print(adjusted_rand_score(adata.obs["kmeans"], adata.obs[config.name_truth]))


# prompt
Important !!!!!!!!! name of niche

In [ ]:
cluster_centers.Tac2


In [ ]:
# manually decide: use "domain_mapping"
# the order of domain_mapping should matches the order of cluster_centers
domain_mapping = {0: "MPN",
                  1: "PVT",
                  2: "MPA",
                  3: "PV",
                  4: "PVH",
                  5: "fx",
                  6: "V3",
                  7: "BST"}


config.domain_mapping = domain_mapping

config.cell_names = neighbor_normalized_df.columns
config.gene_names = top_genes

config.minimal_f = 0.1

In [ ]:
# calculate prototype
one_shot_df = cluster_centers.copy()
print(one_shot_df.index)

# change the index of one_shot_df to be the same as the domain_mapping
one_shot_df.index = one_shot_df.index.map(config.domain_mapping)
print(one_shot_df.index)
# generate Comparison-based Prompt
config.oneshot_prompt = prompt.CP_celltype_geneorder(one_shot_df, config)

In [ ]:
x = [i for i in range(len(adata)) if adata.obs[config.name_truth].iloc[i] == "MPA"][30:40]
print(config.oneshot_prompt + prompt.oneshot_celltype_geneorder(neighbor_normalized_df, neighbor_normalized_df_genes, x, config))



# GPT

## generate json

In [ ]:
config.replicate = "_rep3"

In [ ]:
generate_json_end2end(neighbor_normalized_df, config, prompt.oneshot_celltype_geneorder, batch_size = 3000, n_rows = 1, df_extra= neighbor_normalized_df_genes)



## submit

In [ ]:
# submit_end2end.py
# nohup python -u -m src.submit_end2end configs/config_BZ9_zeroshot.yaml > BZ9_zeroshot.out 2>&1 &
import subprocess
cmd = f"nohup python -u -m src.submit_end2end configs/config_cluster_merfish.yaml MERFISH_27 _rep3 > outs/cluster_merfish_rep3.out 2>&1 &"
subprocess.run(cmd, check=True, text=True, shell=True)


In [ ]:
batch_id = "batch_671ec212c1d08190945b7519d30f8962"
file_response = client.files.content(client.batches.retrieve(batch_id).output_file_id)
save_name = f"response_{config.data_name}_1_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
output_file_name = f"{config.output_path}/{save_name}"
# Open the file in write mode and save the string
with open(output_file_name, 'w') as file:
    file.write(file_response.text)  


In [ ]:
# 初始化空列表以保存custom_id和content
gpt_results_df = pd.DataFrame()
n_batch = 2
for i in range(1,n_batch+1):
    save_name = f"response_{config.data_name}_{i}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.txt"
    output_file_name = f"{config.output_path}/{save_name}"
    # 打开文件并逐行读取
    with open(output_file_name, 'r', encoding='utf-8') as file:
        for line in file:
            try:
                # 解析每一行的json字符串
                json_data = json.loads(line.strip())
                
                # 提取custom_id和content信息
                custom_id = json_data['custom_id']
                content = json_data['response']['body']['choices'][0]['message']['content']
                content = content.replace('\n-', " ").replace('```', "").replace('json', "").replace('plaintext', '').replace('python', '').replace('Output:', 'Outputs:').replace('\n', " ")
                content = content.replace("Layer ", "Layer")
                # extract outputs
                extract_dict = extract_output_microenvironments(content)
                # extract_dict = extract_last_braces(content)

                # 将提取到的信息添加到数据框中
                gpt_results_df = pd.concat([gpt_results_df, pd.DataFrame(extract_dict.values(), index=[custom_id])], ignore_index=False)
                    
            except json.JSONDecodeError:
                print(f"无法解析JSON字符串: {line}")


In [ ]:
gpt_results_df.columns = ['cluster_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)
gpt_results_df.index = gpt_results_df.index.str.replace("id_", "")

In [ ]:
gpt_results_df

In [ ]:
gpt_results_df = gpt_results_df[[0]].copy()

gpt_results_df.columns = ['cluster_gpt4o_mini']
gpt_results_df.index = gpt_results_df.index.astype(str)

In [ ]:
gpt_results_df.index.difference(adata.obs.index)

In [ ]:
gpt_results_df.value_counts()

# Gemini

In [ ]:
import google.generativeai as genai
import pickle
import time



genai.configure(api_key=os.environ["API_KEY"])
model = genai.GenerativeModel("gemini-1.5-pro")
gen_config=genai.types.GenerationConfig(temperature=1.0, max_output_tokens=1000)


In [ ]:
gemini_results_df, store_responses = run_gemini(model, gen_config, 
                                                neighbor_normalized_df, config, 
                                                prompt.oneshot_celltype_geneorder,
                                                df_extra = neighbor_normalized_df_genes)

gemini_results_df.columns = ["cluster_gemini"]

with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'wb') as file:
    pickle.dump(store_responses, file)

gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")


In [ ]:
# find the index of "xxx"
[i for i in range(len(store_responses)) if "V3\n" in store_responses[i]]



In [ ]:
with open(f'./gemini_results/{config.data_name}_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.pkl', 'rb') as file:
    store_responses = pickle.load(file)


In [ ]:
len(store_responses)

In [ ]:
len(neighbor_normalized_df)

In [ ]:
gemini_results_df = pd.read_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv", index_col=0)

In [ ]:
len(gemini_results_df)

In [ ]:
gemini_results_df.loc["552", "cluster_gemini"] = "Layer 5"

In [ ]:
gemini_results_df.index = gemini_results_df.index.astype(str)

In [ ]:
# which index number of gemini_results_df.index == '1'
# [i for i in range(len(gemini_results_df)) if gemini_results_df.index[i] == '1']
gemini_results_df.value_counts()


## plot

In [ ]:

#adata.obs = adata.obs.join(gemini_results_df)
adata.obs = adata.obs.join(gpt_results_df)
#adata.obs['cluster_gemini'] = adata.obs['cluster_gemini'].fillna("unknown")
adata.obs['cluster_gpt4o_mini'] = adata.obs['cluster_gpt4o_mini'].fillna("unknown")
#sc.pl.scatter(adata, x="x", y="y", color="cluster_gemini", title =  f"cluster_gemini")
sc.pl.scatter(adata, x="x", y="y", color="cluster_gpt4o_mini", title =  f"cluster_gpt4o_mini")


In [ ]:
#print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['cluster_gemini']))
print(adjusted_rand_score(adata.obs[config.name_truth], adata.obs['cluster_gpt4o_mini']))



## save results

In [ ]:
gpt_results_df.to_csv(f"./gpt4omini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")
# gemini_results_df.to_csv(f"./gemini_results/{config.data_name}_results_df_{config.model_type}_{config.use_full_name}_{config.with_self_type}_{config.with_region_name}_{config.Graph_type}_{config.with_negatives}_{config.with_CoT}_{config.with_numbers}_{config.with_domain_name}{config.replicate}.csv")

In [ ]:
config.model_type